# 06 · Bench — 06 Judge integrity and inter-rater agreement

**Ported from `specialist-rag`'s `tests/benchmark/judge_model.py` (the `judge_shares_model_with_pipeline` check) and `tests/clinician_validation/analyze_ratings.py`, branch `origin/fix-completion` at commit `130ff868` (not on `main`).**

Two separate integrity questions a benchmark needs answered, neither of
which anything in the cookbook checked before this notebook: is the judge
grading its own homework, and do two humans who rated the same thing
actually agree? `02-deepeval-metrics.ipynb` defaults its judge to
`gpt-4o` with nothing stopping a contributor from pointing the pipeline at
`gpt-4o` too -- a benchmark scored by the system under test is not a
benchmark. And the cookbook had no human-rating analysis path at all.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `judge_shares_model_with_pipeline` | Warns when the judge model and the pipeline's own model are the same | `judge_shares_model_with_pipeline("gpt-4o", "gpt-4o")` |
| `get_judge_model` / `has_judge_model` | Picks whichever provider (OpenAI, Gemini, xAI, Groq) has a key set | `get_judge_model()` |
| `compute_icc` | ICC(2,1) -- do two raters who scored the same subjects agree? | `compute_icc(rater_1_scores, rater_2_scores)` |
| `analyze_ratings` | Full inter-rater report: mean, SD, ICC, safety flags, per metric | `analyze_ratings("ratings.json")` |


In [ ]:
import sys
from pathlib import Path

_p = Path.cwd().resolve()
for _ in range(6):
    if (_p / "nbio.py").is_file():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")

import nbio

repo_root = nbio.bootstrap()
nbio.show_environment()

BENCH_DIR = repo_root / "04-benchmarks" / "clinical-retrieval"
sys.path.insert(0, str(BENCH_DIR))
sys.path.insert(0, str(BENCH_DIR / "clinician_validation"))

## Step 1 — the guardrail, proven first: same model, judge and pipeline

Before anything else in this notebook runs, prove the one check that
matters most: if the model that answered a question and the model
grading it are the same, that's flagged, not silently scored. This is
generalized from the donor's own version, which hardcoded a check for a
"supervisor" concept this cookbook doesn't have -- the underlying
question (is the judge the same model under test?) is unchanged.

In [ ]:
from judge_model import judge_shares_model_with_pipeline

same_model = judge_shares_model_with_pipeline("gpt-4o", "gpt-4o")
different_models = judge_shares_model_with_pipeline("llama-3.1-8b-instant", "gpt-4o")
nothing_to_compare = judge_shares_model_with_pipeline("", "gpt-4o")

print("same model:         ", same_model)
print("different models:   ", different_models)
print("nothing to compare: ", nothing_to_compare)

assert same_model is not None and "gpt-4o" in same_model
assert different_models is None
assert nothing_to_compare is None
print()
print("confirmed: the conflict is caught, and only when there IS one")

## Step 2 — `get_judge_model` / `has_judge_model`: provider selection, offline-checkable

No key is required to prove the selection LOGIC works -- `nbio.show_environment()`
above already reports every key as missing in this environment, so
`has_judge_model()` should report `False` and `get_judge_model()` should
return `None`, gracefully, not raise.

In [ ]:
from judge_model import get_judge_model, has_judge_model, deepeval_installed

print("deepeval installed:", deepeval_installed())
print("has a judge model key:", has_judge_model())
print("resolved judge:", get_judge_model())

if not has_judge_model():
    assert get_judge_model() is None

## Step 3 — `compute_icc`: do two raters agree, or just average out?

Ported verbatim (ICC(2,1), Shrout & Fleiss 1979 case 2). Two raters who
agree closely on every subject should score near `1.0`; two raters who
each individually vary a lot but happen to average out should NOT --
that's the property a plain correlation or a simple mean-difference would
miss, and the reason this needs its own computation rather than a
one-line summary stat.

In [ ]:
from analyze_ratings import compute_icc

agreeing_rater_1 = [4, 5, 3, 4, 5]
agreeing_rater_2 = [4, 5, 3, 4, 4]  # nearly identical per-subject scores

disagreeing_rater_1 = [5, 1, 5, 1, 5]
disagreeing_rater_2 = [1, 5, 1, 5, 1]  # same mean, opposite per-subject pattern

icc_agreeing = compute_icc(agreeing_rater_1, agreeing_rater_2)
icc_disagreeing = compute_icc(disagreeing_rater_1, disagreeing_rater_2)

print(f"close per-subject agreement:    ICC = {icc_agreeing:.3f}")
print(f"same mean, opposite per-subject: ICC = {icc_disagreeing:.3f}")

assert icc_agreeing > 0.8
assert icc_disagreeing < 0
print()
print("confirmed: ICC catches the disagreement a shared mean would hide")

## Step 4 — `analyze_ratings`: the full report, on the shipped template

Runs against `clinician_validation/ratings_template.json` (extended here
with a second question so ICC has more than one subject to compare) --
mean, SD, median, per-rater mean, ICC, and the fraction of scores at 4 or
5, per metric, plus a safety-flag summary.

In [ ]:
import json
import tempfile

from analyze_ratings import analyze_ratings

template_path = BENCH_DIR / "clinician_validation" / "ratings_template.json"
template = json.loads(template_path.read_text())

# The shipped template has one subject (B01) -- ICC needs at least two to be
# defined. Add a second, synthetic subject here so this demo can show a real
# ICC number; never edit the shipped template itself.
template["rater_1"].append({"query_id": "W01", "accuracy": 5, "grounding": 4, "utility": 5, "safety": "Safe"})
template["rater_2"].append({"query_id": "W01", "accuracy": 4, "grounding": 4, "utility": 4, "safety": "Safe"})

tmp_path = Path(tempfile.mkdtemp(prefix="cookbook-ratings-")) / "ratings.json"
tmp_path.write_text(json.dumps(template))

report = analyze_ratings(str(tmp_path))
nbio.show_json(report)

## Where this runs in the pipeline

`judge_shares_model_with_pipeline` is already wired into
`04-benchmarks/clinical-retrieval/eval.py`'s `run_deepeval_metrics` -- set
`EVAL_PIPELINE_MODEL` to whatever model your retrieval/generation backend
actually answers with, and a real run will print a warning if it matches
the resolved judge. `analyze_ratings.py` is not wired into `eval.py` at
all -- it's a separate, offline path for whenever a second person
hand-rates the same answers, run directly:
`python clinician_validation/analyze_ratings.py ratings.json`.